# ECG-CF — calibration sur PTB-XL réel (réponses aux relectures)
Ce notebook produit les chiffres qui manquent au papier, **sans GPU** :

| Section | Question du relecteur | Où dans le papier |
|---|---|---|
| A–B | Les niveaux AF sont-ils réalistes ? (RR CV, ondes f) | 3.3 |
| C | Le détecteur d'ondes P (PLI) est-il calibré ? ROC NORM vs AFIB | 3.2 |
| D | Biais de sélection résiduel : ECG inclus vs exclus | Supplément |
| E | Nos mesures QRS/QT vs l'algorithme clinique GE 12SL (PTB-XL+) | 3.6 |
| F | Contamination : les ECG du fold 10 sont-ils dans les données d'entraînement (ECGInstruct, ECG-R1, GEM) ? | 5 |

La calibration (A–C, E) n'utilise **que les folds 1–9** : rien n'est réglé sur le fold 10 d'où viennent les ECG testés.

**Avant de lancer :**
1. *Settings → Accelerator : None* (CPU suffit) ; *Internet : On*.
2. *Add Input* : **PTB-XL 1.0.3** (la même copie que d'habitude) et, si possible, **PTB-XL+**
   (chercher « ptb-xl+ » ou « ptbxl plus » ; il faut le fichier `12sl_features.csv`).
3. **Save Version → Save & Run All (Commit)**. Les résultats restent dans l'onglet *Output*.
   Durée estimée : 30 à 45 min (le téléchargement d'ECGInstruct fait 1,3 Go).
4. À la fin, envoie-moi la sortie de la **dernière cellule** et `calib_resultats.zip`.

In [ ]:
!pip install -q neurokit2 wfdb huggingface_hub

## 1. Code de l'outil (intégré)

In [ ]:
%%writefile ecg_cf.py
"""ecg_cf — physiology-grounded counterfactual perturbations of 12-lead ECGs.

Pipeline
    fid  = delineate(sig, fs)                       # fiducial points per beat
    sig2, info = perturb(sig, fs, fid, kind, level) # controlled edit of one wave
    ver  = verify(sig, sig2, fs, kind, level)       # independent re-delineation + measurement

Signals: array (n_samples, 12) in millivolts, lead order
I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6 (PTB-XL order).
"""
import numpy as np
from scipy.signal import butter, filtfilt

LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
L = {name: i for i, name in enumerate(LEADS)}

# ----------------------------------------------------------------------------------
# Perturbation catalogue: 5 pathological edits + 3 controls, 3 intensity levels each
# (4 for qrs_wide)
# ----------------------------------------------------------------------------------
CATALOGUE = {
    "st_elev":  dict(levels=[0.10, 0.20, 0.30], unit="mV", wave="ST",
                     dx="anterior ST elevation (STEMI pattern)", leads=["V1", "V2", "V3", "V4"]),
    "t_inv":    dict(levels=[0.5, 1.0, 1.5], unit="x T amplitude", wave="T",
                     dx="T-wave inversion (ischaemia pattern)", leads=["V2", "V3", "V4", "V5"]),
    "qt_long":  dict(levels=[60, 100, 140], unit="ms", wave="QT",
                     dx="prolonged QT interval", leads=None),
    # 4th level (+80 ms) added after the real-data audit: with the conservative
    # independent re-measurement (~0.78x of the stretch), +60 ms met QRS >= 120 ms in only
    # 60% of real recordings, which left the upper end of the dose-response unanchored.
    "qrs_wide": dict(levels=[20, 40, 60, 80], unit="ms", wave="QRS",
                     dx="wide QRS (conduction delay / bundle branch block)", leads=None,
                     # QT includes the QRS: widening it lengthens QT by construction.
                     # This is an expected physiological consequence, not a side effect.
                     secondary=["long_QT"]),
    "af":       dict(levels=[1, 2, 3], unit="severity", wave="P/RR",
                     dx="atrial fibrillation (absent P, irregular RR)", leads=None),
    "ctrl_amp":    dict(levels=[0.85, 1.15, 1.30], unit="x gain", wave=None, dx="control: no change", leads=None),
    "ctrl_noise":  dict(levels=[0.01, 0.02, 0.04], unit="mV sd in TP", wave=None, dx="control: no change", leads=None),
    "ctrl_wander": dict(levels=[0.05, 0.10, 0.20], unit="mV", wave=None, dx="control: no change", leads=None),
}
# (f-wave mV, target RR CV). Evidence levels calibrated on 300 real PTB-XL AFIB recordings
# (folds 1-9): measured RR CV and V1 4-9 Hz activity of the edits match the 25th/50th/75th
# percentiles of real AF (RR CV 0.137/0.182/0.221; V1 activity 0.086/0.132/0.204 mV).
# Level-1 f-wave activity is set by the recording's own noise floor (~0.10 mV, 36th pct);
# the level-3 amplitude is extrapolated beyond the calibration grid (max 0.15).
AF_SEVERITY = {1: (0.02, 0.124), 2: (0.11, 0.164), 3: (0.21, 0.202)}

# ---------------- clinical criteria (guideline-based) ----------------
# ST elevation at the J point, Fourth Universal Definition of MI (Thygesen et al., 2018):
#   >= 0.1 mV in 2 contiguous leads, except V2-V3: >= 0.2 mV men >= 40 y,
#   >= 0.25 mV men < 40 y, >= 0.15 mV women.
# ST depression: >= 0.05 mV in 2 contiguous leads (same source).
# Long QT: QTc (Bazett) >= 450 ms men, >= 460 ms women (AHA/ACCF/HRS 2009 standardisation).
# Contiguity: precordial V1..V6 and the Cabrera limb sequence aVL, I, -aVR, II, aVF, III.
ST_ELEV_MV = 0.10
ST_DEP_MV = -0.05
T_INV_MV = -0.10
QRS_WIDE_MS = 120
RR_CV_AF = 0.08
P_ABSENT_FRAC = 0.30
PHYSIO_HZ = 40          # edits act on the <40 Hz component; the noise residual is kept


def st_threshold(lead, sex=None, age=None):
    if lead not in ("V2", "V3"):
        return ST_ELEV_MV
    if sex == "F":
        return 0.15
    if sex == "M" and age is not None and age < 40:
        return 0.25
    return 0.20                                   # men >= 40 y, or unknown (conservative)


def qtc_threshold(sex=None):
    return 450 if sex == "M" else 460             # unknown sex -> the higher threshold


def _ms(fs, ms):
    return int(round(ms * fs / 1000.0))


def _filt(x, fs, lo=None, hi=None, order=2):
    nyq = fs / 2
    if lo and hi:
        b, a = butter(order, [lo / nyq, hi / nyq], btype="band")
    elif lo:
        b, a = butter(order, lo / nyq, btype="high")
    else:
        b, a = butter(order, hi / nyq, btype="low")
    return filtfilt(b, a, x, axis=0)


def _smooth(v, w):
    w = max(int(w), 1)
    return np.convolve(v, np.ones(w) / w, mode="same")


# ----------------------------------------------------------------------------------
# 1. Delineation (global, multi-lead)
# ----------------------------------------------------------------------------------
def detect_r(sig, fs):
    """R peaks from the spatial derivative energy of all 12 leads (polarity independent)."""
    xf = _filt(sig, fs, 0.5, 40)
    d = _smooth(np.sqrt((np.diff(xf, axis=0, prepend=xf[:1]) ** 2).sum(1)), _ms(fs, 20))
    thr = 0.35 * np.percentile(d, 99)
    refr = _ms(fs, 250)
    peaks, i, n = [], 0, len(d)
    while i < n:
        if d[i] > thr:
            j = min(i + _ms(fs, 120), n)
            k = i + int(np.argmax(d[i:j]))
            peaks.append(k)
            i = k + refr
        else:
            i += 1
    # refine to the extreme of the vector magnitude of the baseline-free signal
    mag = np.sqrt((xf ** 2).sum(1))
    out = []
    for k in peaks:
        a, b = max(k - _ms(fs, 60), 0), min(k + _ms(fs, 60), n)
        out.append(a + int(np.argmax(mag[a:b])))
    return np.array(sorted(set(out)), dtype=int)


def delineate(sig, fs, r=None):
    """Return dict of per-beat fiducials (sample indices, NaN if not found).

    Keys: R, QRS_on, QRS_off (=J), T_on, T_peak, T_off, P_on, P_peak, P_off.
    Uses global (all-lead) derivative and magnitude curves, like the Uni-G/12SL
    global-measurement approach, so it is independent of any single lead's polarity.
    """
    n = sig.shape[0]
    xf = _filt(sig, fs, 0.5, 40)
    r = detect_r(sig, fs) if r is None else np.asarray(r, int)
    d = _smooth(np.sqrt((np.diff(xf, axis=0, prepend=xf[:1]) ** 2).sum(1)), _ms(fs, 16))
    keys = ["R", "QRS_on", "QRS_off", "T_on", "T_peak", "T_off", "P_on", "P_peak", "P_off"]
    F = {k: np.full(len(r), np.nan) for k in keys}
    F["R"] = r.astype(float)
    rr = np.diff(r) if len(r) > 1 else np.array([fs])
    rr_med = float(np.median(rr))

    for i, ri in enumerate(r):
        # ---------------- QRS on/off from derivative energy ----------------
        a = max(ri - _ms(fs, 60), 1)
        pk_on = a + int(np.argmax(d[a:ri + 1]))
        b = min(ri + _ms(fs, 90), n - 1)
        pk_off = ri + int(np.argmax(d[ri:b + 1]))
        dpk = d[a:b + 1].max()
        lo_on = d[max(ri - _ms(fs, 260), 0):max(ri - _ms(fs, 80), 1)]
        lo_off = d[min(ri + _ms(fs, 120), n - 1):min(ri + _ms(fs, 260), n)]
        if len(lo_on) < 5 or len(lo_off) < 5:
            continue
        thr_on = np.percentile(lo_on, 20) + 0.12 * (dpk - np.percentile(lo_on, 20))
        thr_off = np.percentile(lo_off, 20) + 0.12 * (dpk - np.percentile(lo_off, 20))
        k = pk_on
        while k > ri - _ms(fs, 160) and k > 0 and d[k] > thr_on:
            k -= 1
        qon = k
        k = pk_off
        while k < ri + _ms(fs, 200) and k < n - 1 and d[k] > thr_off:
            k += 1
        qoff = k
        F["QRS_on"][i], F["QRS_off"][i] = qon, qoff

        # ---------------- baseline-free magnitude for P and T ----------------
        pr0, pr1 = qon - _ms(fs, 30), qon - _ms(fs, 10)
        if pr0 < 0:
            continue
        base = xf[pr0:pr1].mean(0)
        rr_i = rr[i] if i < len(rr) else rr_med
        t_end = min(ri + int(min(0.62 * rr_i, _ms(fs, 560))), n - 1)
        t_beg = qoff + _ms(fs, 60)
        if t_end - t_beg < _ms(fs, 60):
            continue
        seg = xf[t_beg:t_end] - base
        mag = _smooth(np.sqrt((seg ** 2).sum(1)), _ms(fs, 20))
        tp = int(np.argmax(mag))
        m_iso = np.sqrt(((xf[pr0:pr1] - base) ** 2).sum(1)).mean()
        dm = np.gradient(mag)
        # T offset: tangent at steepest descent intersecting the isoelectric level
        k1 = min(tp + _ms(fs, 200), len(mag) - 1)
        if k1 > tp + 2:
            ks = tp + int(np.argmin(dm[tp:k1]))
            if dm[ks] < 0:
                toff = ks + (m_iso - mag[ks]) / dm[ks]
                F["T_off"][i] = t_beg + min(max(toff, ks), len(mag) + _ms(fs, 150))
        # T onset: tangent at steepest ascent before the peak
        if tp > 2:
            ka = int(np.argmax(dm[:tp]))
            if dm[ka] > 0:
                ton = ka - (mag[ka] - m_iso) / dm[ka]
                F["T_on"][i] = t_beg + max(ton, 0)
        F["T_peak"][i] = t_beg + tp
        if not np.isnan(F["T_off"][i]):
            F["T_off"][i] = min(F["T_off"][i], n - 1)

        # ---------------- P wave ----------------
        p_hi = qon - _ms(fs, 35)
        p_lo = qon - _ms(fs, 280)
        if i > 0 and not np.isnan(F["T_off"][i - 1]):
            p_lo = max(p_lo, int(F["T_off"][i - 1]) + _ms(fs, 20))
        p_lo = max(p_lo, 0)
        if p_hi - p_lo < _ms(fs, 60):
            continue
        pseg = xf[p_lo:p_hi] - base
        pm = _smooth(np.sqrt((pseg ** 2).sum(1)), _ms(fs, 20))
        pp = int(np.argmax(pm))
        # noise reference: high-passed residual in the same window
        noise = np.sqrt((_filt(pseg, fs, lo=15) ** 2).sum(1)).std() + 1e-6
        if pm[pp] - m_iso < max(0.06, 4 * noise):  # no P wave
            continue
        lvl = m_iso + 0.25 * (pm[pp] - m_iso)
        a_, b_ = pp, pp
        while a_ > 0 and pm[a_] > lvl:
            a_ -= 1
        while b_ < len(pm) - 1 and pm[b_] > lvl:
            b_ += 1
        F["P_on"][i], F["P_peak"][i], F["P_off"][i] = p_lo + a_, p_lo + pp, p_lo + b_
    return F


# ----------------------------------------------------------------------------------
# 2. Measurements (used for verification)
# ----------------------------------------------------------------------------------
def _pre_qrs_windows(x, fs, F):
    """detrended pre-QRS windows [QRS_on-240 ms, QRS_on-20 ms], beats x time x leads"""
    a_off, b_off = _ms(fs, 240), _ms(fs, 20)
    wins = []
    for q in F["QRS_on"]:
        if np.isnan(q):
            continue
        q = int(q)
        if q - a_off < 0:
            continue
        w = x[q - a_off:q - b_off].copy()
        # remove linear trend per lead (baseline wander)
        u = np.linspace(-1, 1, len(w))[:, None]
        w -= w.mean(0)
        w -= u * (w * u).sum(0) / (u * u).sum()
        wins.append(w)
    return np.array(wins) if len(wins) >= 3 else None


def phase_locking_index(x, fs, F):
    """PLI of the pre-QRS window (see p_wave_presence); nan if < 3 usable beats"""
    W = _pre_qrs_windows(x, fs, F)
    if W is None:
        return np.nan
    return float((W.mean(0) ** 2).sum() / ((W ** 2).sum((1, 2)).mean() + 1e-12))


def p_wave_presence(x, fs, F, corr_thr=0.5, amp_thr=0.04, pli_thr=0.5):
    """Fraction of beats with a sinus P wave.

    A sinus P wave is phase-locked to the QRS: the pre-QRS window
    [QRS_on-240 ms, QRS_on-20 ms] repeats from beat to beat. We build the median
    template across beats (all 12 leads, detrended) and call a beat 'P present' if it
    correlates with the template (r >= corr_thr) and the template holds a real
    deflection (peak-to-peak magnitude >= amp_thr mV). Fibrillatory waves are not
    phase-locked, so the template averages out and correlations drop."""
    W = _pre_qrs_windows(x, fs, F)
    if W is None:
        return np.nan
    # phase-locking index: power of the mean window / mean power of single windows.
    # ~1 for a repeated (sinus) P wave, ~1/N for activity not locked to the QRS (f-waves).
    # Needed because smooth 4-9 Hz f-waves in a 220 ms window have so few degrees of
    # freedom that per-beat correlations alone are fooled by chance (audit v2 finding).
    pli = float((W.mean(0) ** 2).sum() / ((W ** 2).sum((1, 2)).mean() + 1e-12))
    if pli < pli_thr:
        return 0.0
    present = []
    for k in range(len(W)):                  # leave-one-out template (avoids self-correlation)
        tmpl = np.median(np.delete(W, k, 0), 0)
        amp = np.sqrt((tmpl ** 2).sum(1)).max()
        r = np.corrcoef(W[k].ravel(), tmpl.ravel())[0, 1]
        present.append(amp >= amp_thr and r >= corr_thr)
    return float(np.mean(present))


def measure(sig, fs, F=None):
    """Global and per-lead measurements, median over beats."""
    F = delineate(sig, fs) if F is None else F
    x = _filt(sig, fs, hi=40)                 # no high-pass: keeps true ST level
    n = x.shape[0]
    st, stj, tamp, qrs, qt = [], [], [], [], []
    for i in range(len(F["R"])):
        qon, qoff = F["QRS_on"][i], F["QRS_off"][i]
        if np.isnan(qon) or np.isnan(qoff):
            continue
        qon, qoff = int(qon), int(qoff)
        b0, b1 = qon - _ms(fs, 30), qon - _ms(fs, 10)
        j60 = qoff + _ms(fs, 60)
        if b0 < 0 or j60 >= n:
            continue
        base = x[b0:b1].mean(0)
        st.append(x[j60] - base)
        stj.append(x[qoff:qoff + _ms(fs, 20)].mean(0) - base)   # J point (0-20 ms)
        qrs.append((qoff - qon) * 1000 / fs)
        if not np.isnan(F["T_peak"][i]):
            tpk = int(F["T_peak"][i])
            w = _ms(fs, 20)
            tamp.append(x[max(tpk - w, 0):tpk + w].mean(0) - base)
        if not np.isnan(F["T_off"][i]):
            qt.append((F["T_off"][i] - qon) * 1000 / fs)
    rr = np.diff(F["R"]) * 1000 / fs
    rr_med = float(np.median(rr)) if len(rr) else np.nan
    qt_med = float(np.median(qt)) if qt else np.nan
    m = {
        "n_beats": len(F["R"]),
        "hr_bpm": 60000 / rr_med if rr_med == rr_med else np.nan,
        "rr_cv": float(np.std(rr) / np.mean(rr)) if len(rr) > 2 else np.nan,
        "qrs_ms": float(np.median(qrs)) if qrs else np.nan,
        "qt_ms": qt_med,
        "qtc_ms": qt_med / np.sqrt(rr_med / 1000) if qt else np.nan,             # Bazett
        "qtcf_ms": qt_med / np.cbrt(rr_med / 1000) if qt else np.nan,            # Fridericia
        "p_present_frac": p_wave_presence(x, fs, F),
    }
    st_l = np.median(np.array(st), 0) if st else np.full(12, np.nan)
    stj_l = np.median(np.array(stj), 0) if stj else np.full(12, np.nan)
    t_l = np.median(np.array(tamp), 0) if tamp else np.full(12, np.nan)
    for j, name in enumerate(LEADS):
        m[f"st_{name}"] = float(st_l[j])      # J+60 ms
        m[f"stj_{name}"] = float(stj_l[j])    # J point
        m[f"t_{name}"] = float(t_l[j])
    return m


def _contig(flags):
    """max run of consecutive True in an ordered lead group"""
    best = cur = 0
    for f in flags:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best


PRECORDIAL = ["V1", "V2", "V3", "V4", "V5", "V6"]
CABRERA = ["aVL", "I", "-aVR", "II", "aVF", "III"]


def _v(m, key, lead):
    return -m[f"{key}_aVR"] if lead == "-aVR" else m[f"{key}_{lead}"]


def findings(m, sex=None, age=None):
    """Guideline-based criteria from measurements (booleans). sex: 'M'/'F'/None."""
    st_e = max(_contig([_v(m, "stj", l) >= st_threshold(l, sex, age) for l in grp])
               for grp in (PRECORDIAL, CABRERA)) >= 2
    # ST depression must be horizontal or down-sloping (guideline wording): depressed at the
    # J point AND still depressed at J+60 ms. An up-sloping J-point depression is a normal
    # variant (v2 counted it: main source of false off-target findings on real PTB-XL).
    st_d = max(_contig([(_v(m, "stj", l) <= ST_DEP_MV) and (_v(m, "st", l) <= ST_DEP_MV) for l in grp])
               for grp in (PRECORDIAL, CABRERA)) >= 2
    # T may be negative in normals in V1, III (and aVR, handled as -aVR): excluded
    t_i = max(_contig([_v(m, "t", l) <= T_INV_MV for l in grp])
              for grp in (PRECORDIAL[1:], CABRERA[:5])) >= 2
    return {
        "ST_elevation": bool(st_e),
        "ST_depression": bool(st_d),
        "T_inversion": bool(t_i),
        "long_QT": bool(m["qtc_ms"] >= qtc_threshold(sex)),
        "wide_QRS": bool(m["qrs_ms"] >= QRS_WIDE_MS),
        "AF_pattern": bool((m["p_present_frac"] <= P_ABSENT_FRAC) and (m["rr_cv"] >= RR_CV_AF)),
    }


TARGET_FINDING = {"st_elev": "ST_elevation", "t_inv": "T_inversion", "qt_long": "long_QT",
                  "qrs_wide": "wide_QRS", "af": "AF_pattern"}


# ----------------------------------------------------------------------------------
# 3. Perturbations
# ----------------------------------------------------------------------------------
def _valid(F, i, *keys):
    return all(not np.isnan(F[k][i]) for k in keys)


def _half_cos(n, rising=True):
    w = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, max(n, 1)))
    return w if rising else w[::-1]


def _warp(sig, segs):
    """Piecewise-linear time warp. segs: list of (start, end, delta) in samples on the
    original time axis, non-overlapping and sorted; each segment's length changes by
    delta and everything after it shifts. Total delta must be 0 (length preserved)."""
    n = sig.shape[0]
    ko, kn, c = [0.0], [0.0], 0.0
    for s, e, dl in segs:
        ko += [s, e]
        kn += [s + c, e + c + dl]
        c += dl
    ko.append(n - 1.0)
    kn.append(n - 1.0 + c)
    ko, kn = np.array(ko), np.array(kn)
    keep = np.concatenate([[True], np.diff(ko) > 0])
    ko, kn = ko[keep], kn[keep]
    src = np.interp(np.arange(n), kn, ko)
    idx = np.arange(n)
    return np.stack([np.interp(src, idx, sig[:, j]) for j in range(sig.shape[1])], 1)


def _split(sig, fs):
    """physiological component (<PHYSIO_HZ) and noise residual. Edits act on the first;
    the residual (muscle noise, mains, electrode noise) is added back untouched so the
    edited ECG keeps the recording's own noise floor."""
    s = _filt(sig, fs, hi=PHYSIO_HZ, order=4)
    return s, sig - s


def _warp_keep_noise(sig, segs, fs, qrs_windows=()):
    """Time-warp the physiological component only. Inside QRS windows (whose
    high-frequency content belongs to the QRS itself) the residual is warped too."""
    s, r = _split(sig, fs)
    out = _warp(s, segs)
    if not qrs_windows:
        return out + r
    mask = np.zeros((sig.shape[0], 1))
    for a, b in qrs_windows:
        a, b = max(int(a), 0), min(int(b), sig.shape[0] - 1)
        mask[a:b + 1] = 1.0
    mask = _filt(mask, fs, hi=25, order=2).clip(0, 1)      # soft edges
    m_new = _warp(mask, segs)
    return out + (1 - m_new) * r + m_new * _warp(r, segs)


def _next_boundary(F, i, fs, n):
    """End of the TP segment after beat i (just before the next P wave, or QRS if no P)."""
    if i + 1 < len(F["R"]):
        if not np.isnan(F["P_on"][i + 1]):
            return F["P_on"][i + 1] - _ms(fs, 15)
        if not np.isnan(F["QRS_on"][i + 1]):
            return F["QRS_on"][i + 1] - _ms(fs, 60)
        return None
    return n - 1 - _ms(fs, 5)


def st_elevation(sig, fs, F, amp_mv, leads):
    y = sig.copy()
    idx = [L[l] for l in leads]
    ramp = _ms(fs, 20)
    done = 0
    for i in range(len(F["R"])):
        if not _valid(F, i, "QRS_off", "T_peak", "T_off"):
            continue
        # elevation reaches its full value AT the J point (guideline measurement point):
        # ramp over the terminal 20 ms of the QRS, plateau to the T peak, taper to T end
        j, tp, to = int(F["QRS_off"][i]), int(F["T_peak"][i]), int(F["T_off"][i])
        a = j - ramp
        if not (a > 0 and j < tp < to <= len(y) - 1):
            continue
        w = np.zeros(to - a + 1)
        w[:ramp] = _half_cos(ramp, True)
        w[ramp:tp - a] = 1.0
        w[tp - a:] = _half_cos(to - tp + 1, False)
        y[a:to + 1, idx] += amp_mv * w[:, None]
        done += 1
    return y, {"beats_modified": done}


def _iso(sig, fs, F, i):
    """Isoelectric level per lead from the PR segment of beat i (None if unavailable)."""
    if np.isnan(F["QRS_on"][i]):
        return None
    q = int(F["QRS_on"][i])
    a, b = q - _ms(fs, 30), q - _ms(fs, 10)
    return sig[a:b].mean(0) if a >= 0 else None


def t_inversion(sig, fs, F, factor, leads):
    """Mirror the T wave: T -> -factor * T, on the physiological component.
    Reference line: from the ST level at the start of the edit window to the
    isoelectric level after the T wave (next beat's PR segment interpolated, robust to
    wander). Anchoring on the ST level leaves the ST segment unchanged, so no
    artificial ST depression is created when the ST segment is elevated (v1 bug)."""
    s, res = _split(sig, fs)
    sig = s
    y = s.copy()
    n = len(y)
    idx = [L[l] for l in leads]
    done = 0
    for i in range(len(F["R"])):
        if not _valid(F, i, "QRS_off", "T_on", "T_off"):
            continue
        iso0 = _iso(sig, fs, F, i)
        if iso0 is None:
            continue
        iso1 = _iso(sig, fs, F, i + 1) if i + 1 < len(F["R"]) else None
        iso1 = iso0 if iso1 is None else iso1
        a = max(int(F["T_on"][i]) - _ms(fs, 30), int(F["QRS_off"][i]) + _ms(fs, 20))
        c = _next_boundary(F, i, fs, n)
        b = int(F["T_off"][i]) + _ms(fs, 50)
        if c is not None:
            b = min(b, int(c))
        b = min(b, n - 1)
        if b - a < _ms(fs, 100):
            continue
        tap = _ms(fs, 30)
        w = np.ones(b - a + 1)
        w[:tap] = _half_cos(tap, True)
        w[-tap:] = _half_cos(tap, False)
        q0 = int(F["QRS_on"][i])
        q1 = int(F["QRS_on"][i + 1]) if (i + 1 < len(F["R"]) and not np.isnan(F["QRS_on"][i + 1])) else q0 + n
        # isoelectric level at the end of the window (interpolated between PR segments)
        u_b = (b - q0) / max(q1 - q0, 1)
        iso_b = iso0 + (iso1 - iso0) * u_b
        st_a = sig[max(a - _ms(fs, 5), 0):a + _ms(fs, 5)].mean(0)      # ST level at start
        v = np.linspace(0, 1, b - a + 1)[:, None]
        ref = st_a[None, :] + (iso_b - st_a)[None, :] * v
        for j in idx:
            dev = sig[a:b + 1, j] - ref[:, j]
            y[a:b + 1, j] = sig[a:b + 1, j] - (1 + factor) * w * dev
        done += 1
    return y + res, {"beats_modified": done}


def _stretch(sig, fs, F, delta_ms, start_key, end_key, min_tp_ms=30):
    n = sig.shape[0]
    dl = _ms(fs, delta_ms)
    segs, got = [], []
    for i in range(len(F["R"])):
        if not _valid(F, i, start_key, end_key, "T_off"):
            continue
        s, e, toff = F[start_key][i], F[end_key][i], F["T_off"][i]
        c = _next_boundary(F, i, fs, n)
        if c is None or c <= toff:
            continue
        d = min(dl, (c - toff) - _ms(fs, min_tp_ms))
        if d <= 0:
            continue
        segs += [(s, e, d), (toff, c, -d)]
        got.append(d * 1000 / fs)
    segs.sort(key=lambda t: t[0])
    # drop overlapping segments (e.g. when end_key == T_off)
    clean, last_end = [], -1
    for s, e, d in segs:
        if s >= last_end:
            clean.append((s, e, d))
            last_end = e
    return _warp_keep_noise(sig, clean, fs), {"beats_modified": len(got),
                                              "achieved_ms_mean": float(np.mean(got)) if got else 0.0}


def qt_prolong(sig, fs, F, delta_ms):
    return _stretch(sig, fs, F, delta_ms, "QRS_off", "T_off")


def qrs_widen(sig, fs, F, delta_ms):
    # stretch QRS, shift ST-T, compress the following TP segment
    n = sig.shape[0]
    dl = _ms(fs, delta_ms)
    segs, got = [], []
    for i in range(len(F["R"])):
        if not _valid(F, i, "QRS_on", "QRS_off", "T_off"):
            continue
        c = _next_boundary(F, i, fs, n)
        if c is None or c <= F["T_off"][i]:
            continue
        d = min(dl, (c - F["T_off"][i]) - _ms(fs, 30))
        if d <= 0:
            continue
        # include a 12 ms margin on both sides so the low-slope QRS tails are stretched too
        s0 = F["QRS_on"][i] - _ms(fs, 12)
        e0 = F["QRS_off"][i] + _ms(fs, 12)
        d = int(round(d * (e0 - s0) / (F["QRS_off"][i] - F["QRS_on"][i])))
        d = min(d, int((c - F["T_off"][i]) - _ms(fs, 30)))
        segs += [(s0, e0, d), (F["T_off"][i], c, -d)]
        got.append(d * (F["QRS_off"][i] - F["QRS_on"][i]) / (e0 - s0) * 1000 / fs)
    segs.sort(key=lambda t: t[0])
    qrs_win = [(s0 - _ms(fs, 10), e0 + _ms(fs, 10)) for s0, e0, d in segs if d > 0]
    return _warp_keep_noise(sig, segs, fs, qrs_win), {
        "beats_modified": len(got), "achieved_ms_mean": float(np.mean(got)) if got else 0.0}


def atrial_fibrillation(sig, fs, F, severity, rng):
    f_amp, cv = AF_SEVERITY[int(severity)]
    n = sig.shape[0]
    s, res = _split(sig, fs)
    y = s.copy()
    # (1) remove P waves from the physiological component (smooth bridge); the recording's
    #     own noise residual is kept, so no synthetic noise is injected (v1 injected 1.5x)
    removed = 0
    for i in range(len(F["R"])):
        if not _valid(F, i, "P_on", "P_off", "QRS_on"):
            continue
        a = max(int(F["P_on"][i]) - _ms(fs, 15), 0)
        b = min(int(F["P_off"][i]) + _ms(fs, 15), int(F["QRS_on"][i]) - _ms(fs, 5))
        if b - a < 5:
            continue
        u = _half_cos(b - a + 1, True)[:, None]
        y[a:b + 1] = y[a] + (y[b] - y[a]) * u
        removed += 1
    y = y + res
    # (2) irregular RR: stretch/compress the (now flat) T_off -> next QRS_on segments
    segs = []
    rr_mean = np.median(np.diff(F["R"])) if len(F["R"]) > 1 else fs
    for i in range(len(F["R"]) - 1):
        if not (_valid(F, i, "T_off") and not np.isnan(F["QRS_on"][i + 1])):
            continue
        s, e = F["T_off"][i] + _ms(fs, 10), F["QRS_on"][i + 1] - _ms(fs, 20)
        if e - s > _ms(fs, 60):
            segs.append([s, e, 0.0])
    if len(segs) >= 3:
        lens = np.array([e - s for s, e, _ in segs])
        lo = _ms(fs, 50) - lens                    # keep >= 50 ms of baseline
        d = rng.standard_normal(len(segs))
        for _ in range(60):                        # zero-sum deltas, exact target RR CV
            d = d - d.mean()
            d = d * (cv * 1.1 * rr_mean) / (d.std() + 1e-9)
            d = np.maximum(d, lo)
        d -= d.mean()
        for k in range(len(segs)):
            segs[k][2] = float(d[k])
        y = _warp_keep_noise(y, [tuple(sg) for sg in segs], fs)
    # (3) fibrillatory f-waves: irregular band-limited (4-9 Hz) activity, largest in V1
    #     and the inferior leads; f_amp = 95th percentile of |f| in V1
    t = np.arange(n) / fs
    f = _filt(rng.standard_normal(n + 2 * fs), fs, 4, 9, order=3)[fs:fs + n]
    f *= 1 + 0.3 * np.sin(2 * np.pi * rng.uniform(0.1, 0.4) * t)
    f = f_amp * f / np.percentile(np.abs(f), 95)
    wts = np.array([0.4, 0.7, 0.7, -0.5, -0.2, 0.7, 1.0, 0.8, 0.5, 0.4, 0.3, 0.3])
    y += f[:, None] * wts[None, :]
    return y, {"p_waves_removed": removed, "rr_segments_warped": len(segs)}


def control(sig, fs, F, kind, level, rng):
    n = sig.shape[0]
    if kind == "ctrl_amp":
        return sig * level, {}
    if kind == "ctrl_wander":
        t = np.arange(n) / fs
        ph = rng.uniform(0, 6.28, 12)
        return sig + level * np.sin(2 * np.pi * 0.25 * t[:, None] + ph[None, :]), {}
    if kind == "ctrl_noise":
        y = sig.copy()
        for i in range(len(F["R"])):
            if not _valid(F, i, "T_off"):
                continue
            c = _next_boundary(F, i, fs, n)
            if c is None:
                continue
            a, b = int(F["T_off"][i]) + _ms(fs, 20), int(c) - _ms(fs, 20)
            if b > a:
                y[a:b] += level * rng.standard_normal((b - a, 12))
        return y, {}
    raise ValueError(kind)


def perturb(sig, fs, F, kind, level, seed=0):
    rng = np.random.default_rng(seed)
    spec = CATALOGUE[kind]
    if kind == "st_elev":
        return st_elevation(sig, fs, F, level, spec["leads"])
    if kind == "t_inv":
        return t_inversion(sig, fs, F, level, spec["leads"])
    if kind == "qt_long":
        return qt_prolong(sig, fs, F, level)
    if kind == "qrs_wide":
        return qrs_widen(sig, fs, F, level)
    if kind == "af":
        return atrial_fibrillation(sig, fs, F, level, rng)
    return control(sig, fs, F, kind, level, rng)


# ----------------------------------------------------------------------------------
# 4. Verification: re-delineate the edited signal from scratch and re-measure
# ----------------------------------------------------------------------------------
def verify(m0, sig1, fs, kind, level, sex=None, age=None, achieved=None):
    """m0: measurements of the original. Returns dict with measurement deltas,
    `impl_ok` (edit implemented as specified), `criterion_met` (guideline threshold of
    the target finding reached for this sex/age), off-target findings, and the final
    `keep` decision with an explicit `reject_reason`.
    achieved: for time-warp edits, the stretch actually applied (ms); it can be smaller
    than requested when the TP segment is too short (fast heart rate)."""
    m1 = measure(sig1, fs)
    f0, f1 = findings(m0, sex, age), findings(m1, sex, age)
    spec = CATALOGUE[kind]
    out = {"kind": kind, "level": level}
    new = {k: (f1[k] and not f0[k]) for k in f1}
    target = TARGET_FINDING.get(kind)
    allowed = set(spec.get("secondary", []))
    out["criterion_met"] = bool(f1[target]) if target else False
    out["off_target_new_findings"] = ",".join(k for k, v in new.items()
                                              if v and k != target and k not in allowed)
    out["expected_secondary_findings"] = ",".join(k for k, v in new.items() if v and k in allowed)
    ach = level if achieved is None else achieved

    def dl(key):
        return m1[key] - m0[key]
    out["d_qrs_ms"], out["d_qt_ms"], out["d_qtc_ms"] = dl("qrs_ms"), dl("qt_ms"), dl("qtc_ms")
    out["d_rr_cv"], out["d_p_present"] = dl("rr_cv"), dl("p_present_frac")
    out["rr_cv"], out["p_present_frac"] = m1["rr_cv"], m1["p_present_frac"]
    out["qtc_ms"], out["qrs_ms"] = m1["qtc_ms"], m1["qrs_ms"]
    if spec["leads"]:
        tl = spec["leads"]
        if kind == "st_elev":
            d = np.array([dl(f"stj_{l}") for l in tl])
            out["d_st_target_mv"] = float(d.mean())
            out["impl_ok"] = bool(abs(d.mean() - level) <= 0.03)
        if kind == "t_inv":
            t0 = np.array([m0[f"t_{l}"] for l in tl])
            t1 = np.array([m1[f"t_{l}"] for l in tl])
            ok = t0 > 0.1
            ratio = (t1[ok] / t0[ok]) if ok.any() else np.array([np.nan])
            out["t_ratio_target"] = float(np.median(ratio))
            out["impl_ok"] = bool(ok.any() and abs(np.median(ratio) + level) <= 0.35 * level)
    elif kind == "qt_long":
        # compare the measurement with the stretch actually applied
        out["impl_ok"] = bool(abs(out["d_qt_ms"] - ach) <= max(20, 0.25 * ach)
                              and abs(out["d_qrs_ms"]) <= 10)
    elif kind == "qrs_wide":
        # the time-warp is exact by construction; the independent re-measurement is
        # conservative for stretched waves (~0.75x), so accept 0.6x-1.2x of the stretch
        out["impl_ok"] = bool(0.6 * ach <= out["d_qrs_ms"] <= 1.2 * ach)
    elif kind == "af":
        out["impl_ok"] = bool(m1["p_present_frac"] <= P_ABSENT_FRAC and m1["rr_cv"] >= RR_CV_AF)
    else:  # controls: must not create any finding nor move key measurements
        # a gain change legitimately scales an existing ST offset: widen tolerance for it
        tol = 0.05 + (abs(level - 1) * max(abs(m0[f"stj_{l}"]) for l in PRECORDIAL)
                      if kind == "ctrl_amp" else 0)
        stab = (abs(out["d_qrs_ms"]) <= 10 and abs(out["d_qt_ms"]) <= 20 and
                max(abs(dl(f"stj_{l}")) for l in PRECORDIAL) <= tol)
        out["impl_ok"] = bool(stab and not any(new.values()))
    # ---- final decision, with an explicit reason (no guessing when analysing rejects)
    reasons = []
    if kind in ("qt_long", "qrs_wide") and ach < 0.8 * level:
        reasons.append(f"capped({ach:.0f}/{level:g}ms)")
    if not out.get("impl_ok", False):
        reasons.append("impl")
    if out["off_target_new_findings"]:
        reasons.append("off_target:" + out["off_target_new_findings"])
    out["achieved"] = ach
    out["reject_reason"] = ";".join(reasons)
    out["keep"] = not reasons
    return out, m1


In [ ]:
%%writefile run_ptbxl.py
"""Build the counterfactual ECG set from PTB-XL.

1. selects clean, human-validated normal sinus-rhythm ECGs (default: test fold 10),
2. applies the 8 edits x 3 levels,
3. re-measures every edited ECG independently and keeps only verified edits,
4. writes the signals (.npz, one per record) and a manifest (.csv) for model querying.

Usage (PTB-XL v1.0.3 from https://physionet.org/content/ptb-xl/1.0.3/):
    python run_ptbxl.py --ptbxl /path/to/ptb-xl-1.0.3 --n 500 --out cf_ptbxl
"""
import argparse
import ast
import os
import time
import warnings

import numpy as np
import pandas as pd
import wfdb

from ecg_cf import CATALOGUE, LEADS, TARGET_FINDING, delineate, findings, measure, perturb, verify

warnings.filterwarnings("ignore")
QUALITY_COLS = ["baseline_drift", "static_noise", "burst_noise", "electrodes_problems",
                "extra_beats", "pacemaker"]


def select_records(db, folds, seed, n):
    codes = db.scp_codes.apply(ast.literal_eval)
    normal = codes.apply(lambda c: c.get("NORM", 0) == 100 and set(c) <= {"NORM", "SR"})
    clean = db[[c for c in QUALITY_COLS if c in db]].isna().all(1)
    ok = normal & clean & (db.validated_by_human == True) & db.strat_fold.isin(folds)  # noqa: E712
    sel = db[ok]
    print(f"{ok.sum()} clean validated NORM records in folds {folds}")
    return sel.sample(frac=1.0, random_state=seed)   # shuffled; we stop once n pass the checks


def infer_sex_coding(db):
    """PTB-XL documents 52% male / 48% female but not the 0/1 coding explicitly.
    Infer it from the data instead of assuming, and print it for the user to confirm."""
    share = db.sex.value_counts(normalize=True)
    male_code = (share - 0.52).abs().idxmin()
    print(f"sex coding inferred from data: {share.round(3).to_dict()} -> male = {male_code}"
          f" (PTB-XL paper: 52% male). Check this once.")
    return lambda v: "M" if v == male_code else ("F" if v == v else None)


def clean_age(a):
    """PTB-XL sets ages >= 90 to 300 (HIPAA)."""
    if a != a:
        return None
    return 90 if a >= 300 else int(a)


def baseline_reason(m0, sex, age, hr=(50, 90)):
    """why a candidate normal ECG is not used as a base recording ("" = included)"""
    if m0["n_beats"] < 6:
        return "too few beats"
    if not (hr[0] <= m0["hr_bpm"] <= hr[1]):
        return f"HR {m0['hr_bpm']:.0f}"
    if not (m0["p_present_frac"] >= 0.8):
        return "P waves not detected"
    f0 = findings(m0, sex, age)
    if any(f0.values()):
        return "baseline measurement not normal: " + ",".join(k for k, v in f0.items() if v)
    return ""


def load(ptbxl, fname):
    sig, meta = wfdb.rdsamp(os.path.join(ptbxl, fname))
    names = [s.upper() for s in meta["sig_name"]]
    order = [names.index(l.upper()) for l in LEADS]
    return sig[:, order].astype(np.float64), int(meta["fs"])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ptbxl", required=True, help="folder containing ptbxl_database.csv")
    ap.add_argument("--n", type=int, default=500, help="number of base ECGs to keep")
    ap.add_argument("--folds", type=int, nargs="+", default=[10],
                    help="PTB-XL strat folds to draw from (10 = recommended test fold)")
    ap.add_argument("--hr", type=float, nargs=2, default=[50, 90],
                    help="heart-rate range (bpm); leaves room for QT/QRS stretching")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--exclude_ids", nargs="*", default=[],
                    help="JSON files with PTB-XL ecg_ids to exclude (e.g. records found in the "
                         "instruction-tuning data of the evaluated models)")
    ap.add_argument("--out", default="cf_ptbxl")
    a = ap.parse_args()

    os.makedirs(os.path.join(a.out, "signals"), exist_ok=True)
    db = pd.read_csv(os.path.join(a.ptbxl, "ptbxl_database.csv"), index_col="ecg_id")
    cand = select_records(db, a.folds, a.seed, a.n)
    excl = set()
    for f in a.exclude_ids:
        excl |= {int(i) for i in __import__("json").load(open(f))}
    if excl:
        before = len(cand)
        cand = cand[~cand.index.isin(excl)]
        print(f"excluded {before - len(cand)} candidates found in training data ({len(cand)} left)")
    to_sex = infer_sex_coding(db)

    base_rows, rows, kept, t0 = [], [], 0, time.time()
    for ecg_id, r in cand.iterrows():
        if kept >= a.n:
            break
        sex, age = to_sex(r.sex), clean_age(r.age)
        try:
            sig, fs = load(a.ptbxl, r.filename_hr)
            F = delineate(sig, fs)
            m0 = measure(sig, fs, F)
        except Exception as e:                       # corrupted / unreadable record
            print(f"  {ecg_id}: skipped ({e})")
            continue
        reason = baseline_reason(m0, sex, age, a.hr)
        base_rows.append(dict(ecg_id=ecg_id, age=age, sex=sex, fold=r.strat_fold,
                              excluded=reason or "", **m0))
        if reason:
            continue
        arrays = {"original": sig.astype(np.float32)}
        for kind, spec in CATALOGUE.items():
            for li, level in enumerate(spec["levels"], start=1):
                y, info = perturb(sig, fs, F, kind, level, seed=int(ecg_id) * 10 + li)
                v, _ = verify(m0, y, fs, kind, level, sex=sex, age=age,
                              achieved=info.get("achieved_ms_mean"))
                sid = f"{ecg_id}_{kind}_L{li}"
                arrays[sid] = y.astype(np.float32)
                rows.append(dict(
                    sample_id=sid, ecg_id=ecg_id, sex=sex, age=age, kind=kind, level=level, level_idx=li,
                    unit=spec["unit"], is_control=kind.startswith("ctrl"),
                    expected_change=spec["dx"], target_wave=spec["wave"] or "",
                    target_leads="-".join(spec["leads"]) if spec["leads"] else "global",
                    target_finding=TARGET_FINDING.get(kind, ""),
                    **{k: v[k] for k in v if k not in ("kind", "level")}, **info))
        np.savez_compressed(os.path.join(a.out, "signals", f"{ecg_id}.npz"), fs=fs, leads=LEADS, **arrays)
        kept += 1
        if kept % 25 == 0:
            print(f"  {kept}/{a.n} base ECGs done ({time.time() - t0:.0f}s)")

    man = pd.DataFrame(rows)
    man.to_csv(os.path.join(a.out, "manifest.csv"), index=False)
    pd.DataFrame(base_rows).to_csv(os.path.join(a.out, "base_ecgs.csv"), index=False)
    s = (man.groupby(["kind", "level_idx", "level"])
            .agg(n=("keep", "size"), kept=("keep", "mean"), criterion_met=("criterion_met", "mean"))
            .reset_index())
    s.to_csv(os.path.join(a.out, "summary.csv"), index=False)
    print(f"\n{kept} base ECGs, {len(man)} edited ECGs, {int(man.keep.sum())} verified")
    print(s.round(3).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile calibrate.py
"""Calibration of ECG-CF against real PTB-XL recordings (answers reviewer items).

All calibration uses folds 1-9 only, so nothing is tuned on the test fold 10 from which
the base recordings are drawn (except the selection-bias table, which describes fold 10).

  A. real_profiles     measurements of real NORM, AFIB and wide-QRS recordings
                       (RR CV, P presence, phase-locking index, f-wave amplitude, QRS, QT)
  B. af_edit_grid      the same measurements on our AF edits over a grid of
                       (f-wave amplitude, target RR CV) -> place edit levels on the real
                       AF distribution
  C. detector ROC      PLI / P presence / RR CV / AF rule, NORM vs AFIB
  D. selection table   fold-10 candidates: included vs excluded (residual selection bias)
  E. 12SL agreement    our QRS/QT vs GE 12SL (PTB-XL+), incl. wide-QRS recordings
"""
import ast
import os
import time

import numpy as np
import pandas as pd

import ecg_cf as E
from run_ptbxl import QUALITY_COLS, baseline_reason, clean_age, infer_sex_coding, load

AF_CODES = {"AFIB"}
WIDE_CODES = {"CLBBB", "CRBBB", "IVCD"}


def scp(db):
    return db.scp_codes.apply(ast.literal_eval)


def pick(db, group, folds, n, seed=0):
    c = scp(db)
    clean_noise = db[[k for k in ["static_noise", "burst_noise", "electrodes_problems"] if k in db]].isna().all(1)
    no_pace = db["pacemaker"].isna() if "pacemaker" in db else True
    base = db.strat_fold.isin(folds) & (db.validated_by_human == True) & no_pace  # noqa: E712
    if group == "NORM":
        ok = c.apply(lambda d: d.get("NORM", 0) == 100 and set(d) <= {"NORM", "SR"})
        ok &= db[[k for k in QUALITY_COLS if k in db]].isna().all(1)
    elif group == "AFIB":
        ok = c.apply(lambda d: bool(AF_CODES & set(d)) and "AFLT" not in d) & clean_noise
    elif group == "WIDE":
        # exclude co-existing AF/flutter so that AF-rule positives on this group are false positives
        ok = c.apply(lambda d: any(d.get(k, 0) >= 80 for k in WIDE_CODES) and not ({"AFIB", "AFLT"} & set(d))) & clean_noise
    else:
        raise ValueError(group)
    sel = db[base & ok]
    return sel.sample(n=min(n, len(sel)), random_state=seed)


def fwave_amp(x, fs, F, lead="V1"):
    """95th percentile of |4-9 Hz activity| in lead V1 during the T-Q intervals.
    Same definition as the f_amp parameter of the AF edit (which scales the 95th
    percentile of the injected f-wave in V1), measured identically on real and edited ECGs."""
    j = E.LEADS.index(lead)
    f = E._filt(x[:, j], fs, 4, 9, order=3)
    n = len(f)
    mask = np.zeros(n, bool)
    R = F["R"]
    for i in range(len(R) - 1):
        a = F["T_off"][i] if not np.isnan(F["T_off"][i]) else R[i] + E._ms(fs, 400)
        b = F["QRS_on"][i + 1] if not np.isnan(F["QRS_on"][i + 1]) else R[i + 1] - E._ms(fs, 60)
        a, b = int(a) + E._ms(fs, 40), int(b) - E._ms(fs, 20)
        if b > a:
            mask[a:b] = True
    if mask.sum() < E._ms(fs, 200):
        return np.nan
    return float(np.percentile(np.abs(f[mask]), 95))


def profile(x, fs):
    F = E.delineate(x, fs)
    m = E.measure(x, fs, F)
    f = E.findings(m)
    return dict(hr_bpm=m["hr_bpm"], rr_cv=m["rr_cv"], p_present_frac=m["p_present_frac"],
                pli=E.phase_locking_index(E._filt(x, fs, hi=40), fs, F), fwave_v1=fwave_amp(x, fs, F),
                qrs_ms=m["qrs_ms"], qt_ms=m["qt_ms"], qtc_ms=m["qtc_ms"], n_beats=m["n_beats"],
                AF_pattern=f["AF_pattern"], wide_QRS=f["wide_QRS"])


def real_profiles(ptbxl, db, folds, n_per_group, seed=0):
    rows = []
    for g in ["NORM", "AFIB", "WIDE"]:
        sel = pick(db, g, folds, n_per_group, seed)
        t0 = time.time()
        for ecg_id, r in sel.iterrows():
            try:
                x, fs = load(ptbxl, r.filename_hr)
                rows.append(dict(ecg_id=ecg_id, group=g, fold=r.strat_fold, **profile(x, fs)))
            except Exception as e:  # unreadable / undelineable record
                rows.append(dict(ecg_id=ecg_id, group=g, fold=r.strat_fold, error=str(e)[:80]))
        print(f"{g}: {len(sel)} records ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows)


def af_edit_grid(ptbxl, db, folds, n_base, cv_grid, f_grid, f_fixed=0.05, cv_fixed=0.15, seed=1):
    """AF edits on real NORM recordings over (f_amp, cv) and the current levels 1-3"""
    sel = pick(db, "NORM", folds, n_base, seed)
    saved = dict(E.AF_SEVERITY)
    configs = [("current", lv, *saved[lv]) for lv in sorted(saved)]
    configs += [("cv_grid", None, f_fixed, cv) for cv in cv_grid]
    configs += [("f_grid", None, fa, cv_fixed) for fa in f_grid]
    rows = []
    try:
        for ecg_id, r in sel.iterrows():
            x, fs = load(ptbxl, r.filename_hr)
            F = E.delineate(x, fs)
            m0 = E.measure(x, fs, F)
            if baseline_reason(m0, None, None):
                continue
            for k, (fam, lv, fa, cv) in enumerate(configs):
                E.AF_SEVERITY[99] = (fa, cv)
                y, _ = E.atrial_fibrillation(x, fs, F, 99, np.random.default_rng(int(ecg_id) * 100 + k))
                rows.append(dict(ecg_id=ecg_id, family=fam, level=lv, f_amp=fa, cv_target=cv, **profile(y, fs)))
    finally:
        E.AF_SEVERITY.clear()
        E.AF_SEVERITY.update(saved)
    return pd.DataFrame(rows)


def roc(neg, pos):
    """AUC (pos scores higher) and the full curve, no sklearn dependency"""
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    neg, pos = neg[~np.isnan(neg)], pos[~np.isnan(pos)]
    thr = np.unique(np.r_[neg, pos, np.inf, -np.inf])
    tpr = np.array([(pos >= t).mean() for t in thr])
    fpr = np.array([(neg >= t).mean() for t in thr])
    o = np.argsort(fpr + 1e-12 * tpr)
    _tz = getattr(np, "trapezoid", None) or np.trapz
    auc = float(_tz(tpr[o], fpr[o]))
    j = np.argmax(tpr - fpr)
    return dict(auc=auc, youden_thr=float(thr[j]), sens=float(tpr[j]), spec=float(1 - fpr[j]),
                n_neg=len(neg), n_pos=len(pos))


def at_threshold(neg, pos, thr, higher_is_pos=True):
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    neg, pos = neg[~np.isnan(neg)], pos[~np.isnan(pos)]
    if higher_is_pos:
        return dict(sens=float((pos >= thr).mean()), spec=float((neg < thr).mean()))
    return dict(sens=float((pos < thr).mean()), spec=float((neg >= thr).mean()))


def selection_table(ptbxl, db, folds=(10,), hr=(50, 90)):
    """all clean validated NORM candidates of the test fold: included vs excluded"""
    to_sex = infer_sex_coding(db)
    c = scp(db)
    normal = c.apply(lambda d: d.get("NORM", 0) == 100 and set(d) <= {"NORM", "SR"})
    clean = db[[k for k in QUALITY_COLS if k in db]].isna().all(1)
    cand = db[normal & clean & (db.validated_by_human == True) & db.strat_fold.isin(folds)]  # noqa: E712
    rows = []
    for ecg_id, r in cand.iterrows():
        sex, age = to_sex(r.sex), clean_age(r.age)
        try:
            x, fs = load(ptbxl, r.filename_hr)
            F = E.delineate(x, fs)
            m0 = E.measure(x, fs, F)
        except Exception as e:
            rows.append(dict(ecg_id=ecg_id, sex=sex, age=age, excluded=f"unreadable ({str(e)[:40]})"))
            continue
        rows.append(dict(ecg_id=ecg_id, sex=sex, age=age, excluded=baseline_reason(m0, sex, age, hr),
                         **{k: m0[k] for k in ["hr_bpm", "qrs_ms", "qtc_ms", "p_present_frac", "stj_V2", "stj_V3",
                                               "t_V2", "t_V5"]}))
    return pd.DataFrame(rows)


def smd(a, b):
    a, b = pd.Series(a, dtype=float).dropna(), pd.Series(b, dtype=float).dropna()
    s = np.sqrt((a.var() + b.var()) / 2)
    return float((a.mean() - b.mean()) / s) if s > 0 else np.nan


def describe_selection(sel):
    inc = sel[sel.excluded == ""]
    exc = sel[sel.excluded != ""]
    rows = []
    for col, lab in [("age", "age (y)"), ("hr_bpm", "heart rate (bpm)"), ("qrs_ms", "QRS (ms)"),
                     ("qtc_ms", "QTc Bazett (ms)"), ("stj_V2", "J-point V2 (mV)"), ("stj_V3", "J-point V3 (mV)"),
                     ("t_V2", "T amplitude V2 (mV)"), ("t_V5", "T amplitude V5 (mV)")]:
        if col not in sel:
            continue
        q = lambda d: f"{d[col].median():.3g} [{d[col].quantile(.25):.3g}-{d[col].quantile(.75):.3g}]"
        rows.append(dict(variable=lab, included=q(inc), excluded=q(exc), SMD=round(smd(inc[col], exc[col]), 2)))
    male = lambda d: (d.sex == "M").mean()
    rows.append(dict(variable="male (%)", included=f"{100 * male(inc):.0f}", excluded=f"{100 * male(exc):.0f}",
                     SMD=round(smd((inc.sex == "M").astype(float), (exc.sex == "M").astype(float)), 2)))
    rows.append(dict(variable="n", included=str(len(inc)), excluded=str(len(exc)), SMD=np.nan))
    reasons = exc.excluded.str.split(":").str[0].str.replace(r"HR \d+", "HR outside 50-90", regex=True)
    return pd.DataFrame(rows), reasons.value_counts()


## 2. Trouver PTB-XL

In [ ]:
import os, glob, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
MOCK = os.environ.get("PILOT_MOCK") == "1"
def find_file(name, root="/kaggle/input", max_depth=8):
    for dirpath, dirnames, files in os.walk(root, followlinks=True):
        if name in files:
            return dirpath
        if dirpath.count(os.sep) - root.count(os.sep) > max_depth:
            dirnames[:] = []
    return None
PTBXL = os.environ.get("PTBXL_DIR") or find_file("ptbxl_database.csv")
assert PTBXL, "PTB-XL introuvable : ajoutez-le via Add Input"
db = pd.read_csv(os.path.join(PTBXL, "ptbxl_database.csv"), index_col="ecg_id")
print("PTB-XL :", PTBXL, "|", len(db), "enregistrements")
CAL_FOLDS = list(range(1, 11)) if MOCK else list(range(1, 10))   # calibration : folds 1-9
OUT = "/kaggle/working/calib"; os.makedirs(OUT, exist_ok=True)
import calibrate as C, ecg_cf as E

## A. Profils des vrais ECG (NORM, AFIB, QRS large), folds 1–9
Pour chaque ECG : variabilité RR (CV), fraction de battements avec onde P, indice de verrouillage de
phase (PLI), amplitude des ondes f en V1 (4–9 Hz, intervalles T–Q), QRS, QT.

In [ ]:
N_PER_GROUP = 300
real = C.real_profiles(PTBXL, db, CAL_FOLDS, N_PER_GROUP)
real.to_csv(f"{OUT}/real_profiles.csv", index=False)
cols = ["hr_bpm", "rr_cv", "p_present_frac", "pli", "fwave_v1", "qrs_ms", "qtc_ms"]
q = real.groupby("group")[cols].quantile([.1, .25, .5, .75, .9]).round(3)
print(q.to_string())

## B. Nos modifications AF placées sur la distribution des vraies AF
On applique l'édition AF à 60 ECG normaux (folds 1–9) : niveaux actuels 1–3, puis une grille de
CV RR cible (ondes f fixées à 0,05 mV) et une grille d'amplitude des ondes f (CV fixé à 0,15).
On mesure le résultat **avec les mêmes fonctions** que pour les vrais ECG.

In [ ]:
N_EDIT_BASE = 60
CV_GRID = [0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
F_GRID = [0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
grid = C.af_edit_grid(PTBXL, db, CAL_FOLDS, N_EDIT_BASE, CV_GRID, F_GRID)
grid.to_csv(f"{OUT}/af_edit_grid.csv", index=False)
af = real[real.group == "AFIB"]; nm = real[real.group == "NORM"]
pct = lambda v, s: float((s.dropna() < v).mean() * 100)
g = (grid.groupby(["family", "level", "f_amp", "cv_target"], dropna=False)
         [["rr_cv", "fwave_v1", "pli", "p_present_frac", "AF_pattern"]].median().reset_index())
g["rr_cv_pct_in_realAF"] = g.rr_cv.apply(lambda v: pct(v, af.rr_cv))
g["fwave_pct_in_realAF"] = g.fwave_v1.apply(lambda v: pct(v, af.fwave_v1))
print(g.round(3).to_string(index=False))
# proposition : niveaux d'évidence = quartiles des vraies AF (25e, 50e, 75e percentiles)
prop = {}
for i, p in enumerate([.25, .5, .75], start=1):
    cv_real, f_real = af.rr_cv.quantile(p), af.fwave_v1.quantile(p)
    # inverser la relation cible -> mesuré obtenue sur la grille
    cvg = g[g.family == "cv_grid"].sort_values("cv_target")
    fg = g[g.family == "f_grid"].sort_values("f_amp")
    cv_t = float(np.interp(cv_real, cvg.rr_cv, cvg.cv_target))
    f_t = float(np.interp(f_real, fg.fwave_v1, fg.f_amp))
    prop[i] = (round(f_t, 3), round(cv_t, 3))
    print(f"niveau {i} = {int(p*100)}e percentile des vraies AF : RR CV {cv_real:.3f}, ondes f {f_real:.3f} mV "
          f"-> paramètres d'édition f_amp={f_t:.3f}, cv={cv_t:.3f}")
print("\nAF_SEVERITY proposé :", prop, "| actuel :", E.AF_SEVERITY)
json.dump({"proposed_AF_SEVERITY": prop}, open(f"{OUT}/af_proposed_levels.json", "w"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for a, col, lab in [(ax[0], "rr_cv", "RR CV"), (ax[1], "fwave_v1", "4–9 Hz activity in V1, 95th pct (mV)")]:
    bins = np.linspace(0, np.nanpercentile(real[col], 99), 40)
    a.hist(nm[col].dropna(), bins, alpha=.55, label="PTB-XL NORM", color="#9b9a95")
    a.hist(af[col].dropna(), bins, alpha=.55, label="PTB-XL AFIB", color="#2a78d6")
    for lv, v in g[g.family == "current"].set_index("level")[col].items():
        a.axvline(v, color="#c0392b", ls="--", lw=1); a.text(v, a.get_ylim()[1]*.9, f" L{int(lv)}", color="#c0392b", fontsize=8)
    a.set_xlabel(lab); a.legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_af_calibration.png", dpi=200); plt.show()

## C. Détecteur d'ondes P et règle AF : ROC NORM vs AFIB (folds 1–9)

In [ ]:
rows = []
neg, pos = nm, af
for name, s_neg, s_pos, thr, hi in [
        ("PLI (low = AF)", -neg.pli, -pos.pli, -0.5, True),
        ("P-present fraction (low = AF)", -neg.p_present_frac, -pos.p_present_frac, -0.30, True),
        ("RR CV (high = AF)", neg.rr_cv, pos.rr_cv, 0.08, True)]:
    r = C.roc(s_neg, s_pos)
    t = C.at_threshold(s_neg, s_pos, thr)
    rows.append(dict(score=name, AUC=r["auc"], youden_thr=abs(r["youden_thr"]), sens_youden=r["sens"],
                     spec_youden=r["spec"], sens_at_ours=t["sens"], spec_at_ours=t["spec"], n_NORM=r["n_neg"], n_AFIB=r["n_pos"]))
rule = lambda d: d.AF_pattern.astype(bool)
rows.append(dict(score="AF rule (P<=30% & CV>=0.08)", sens_at_ours=rule(pos).mean(), spec_at_ours=1 - rule(neg).mean(),
                 n_NORM=len(neg), n_AFIB=len(pos)))
wide = real[real.group == "WIDE"]
rows.append(dict(score="AF rule on wide-QRS (false positives)", spec_at_ours=1 - rule(wide).mean(), n_NORM=len(wide)))
roc_tab = pd.DataFrame(rows); roc_tab.to_csv(f"{OUT}/detector_roc.csv", index=False)
print(roc_tab.round(3).to_string(index=False))

## D. Biais de sélection résiduel (fold 10) : ECG normaux inclus vs exclus
Tous les ECG NORM validés et sans problème de qualité du fold 10 passent par nos critères d'inclusion.
SMD = différence standardisée (|SMD| > 0,2 : différence notable).

In [ ]:
sel = C.selection_table(PTBXL, db, folds=(10,))
sel.to_csv(f"{OUT}/selection_fold10.csv", index=False)
tab, reasons = C.describe_selection(sel)
tab.to_csv(f"{OUT}/selection_table.csv", index=False)
print(tab.to_string(index=False)); print("\nMotifs d'exclusion :"); print(reasons.to_string())

## E. Nos mesures vs GE 12SL (PTB-XL+)
Accord QRS/QT sur les ECG normaux, AF et à QRS large. La pente sur les QRS larges dit si notre mesure
sous-estime les QRS élargis (ce qui expliquerait le facteur ~0,78 observé sur nos étirements).

In [ ]:
d12 = find_file("12sl_features.csv")
if d12 is None and not MOCK:
    # PTB-XL+ absent en Input : téléchargement direct du seul fichier nécessaire depuis PhysioNet
    import urllib.request
    os.makedirs("/kaggle/working/ptbxlplus", exist_ok=True)
    for ver in ["1.0.1", "1.0.0"]:
        try:
            data = urllib.request.urlopen(f"https://physionet.org/files/ptb-xl-plus/{ver}/features/12sl_features.csv",
                                          timeout=60).read()          # 60 s maximum : jamais de blocage
            open("/kaggle/working/ptbxlplus/12sl_features.csv", "wb").write(data)
            d12 = "/kaggle/working/ptbxlplus"; print("PTB-XL+", ver, "téléchargé depuis PhysioNet"); break
        except Exception as e:
            print("téléchargement PTB-XL+", ver, "impossible :", e)
if d12 is None:
    print("PTB-XL+ absent : section E sautée (ajoutez PTB-XL+ en Input pour l'obtenir).")
    agree = None
else:
    sl = pd.read_csv(os.path.join(d12, "12sl_features.csv"))
    idc = [c for c in sl.columns if c.lower() in ("ecg_id", "ecgid")][0]
    qrs_c = [c for c in sl.columns if re.search(r"qrs", c, re.I) and re.search(r"dur", c, re.I)]
    qt_c = [c for c in sl.columns if re.match(r"qt(?!c)", c, re.I) and not re.search(r"disp", c, re.I)]
    print("colonnes 12SL utilisées :", qrs_c[:1], qt_c[:1], "| autres candidates :", qrs_c[1:4], qt_c[1:4])
    both = pd.concat([real[["ecg_id", "group", "qrs_ms", "qt_ms"]],
                      sel.assign(group="NORM_fold10")[["ecg_id", "group", "qrs_ms"]]])
    mm = both.merge(sl[[idc, qrs_c[0], qt_c[0]]], left_on="ecg_id", right_on=idc)
    rows = []
    for (grp, mine, theirs) in [(g_, a_, b_) for g_ in mm.group.unique() for a_, b_ in [("qrs_ms", qrs_c[0]), ("qt_ms", qt_c[0])]]:
        d = mm[mm.group == grp][[mine, theirs]].dropna()
        if len(d) < 5: continue
        diff = d[mine] - d[theirs]
        slope = np.polyfit(d[theirs], d[mine], 1)[0]
        rows.append(dict(group=grp, measure=mine, n=len(d), r=d.corr().iloc[0, 1], bias_ms=diff.mean(),
                         loa_low=diff.mean() - 1.96 * diff.std(), loa_high=diff.mean() + 1.96 * diff.std(), slope=slope,
                         ours_ge120=(d[mine] >= 120).mean() if mine == "qrs_ms" else np.nan,
                         ge12sl_ge120=(d[theirs] >= 120).mean() if mine == "qrs_ms" else np.nan))
    agree = pd.DataFrame(rows); agree.to_csv(f"{OUT}/agreement_12sl.csv", index=False)
    print(agree.round(3).to_string(index=False))

## F. Contamination : ECG PTB-XL présents dans les données d'entraînement
On télécharge les fichiers d'instructions publics (JSON uniquement, pas les images) et on cherche les
identifiants PTB-XL (`NNNNN_hr` / `NNNNN_lr`). Puis : combien appartiennent au fold 10 ? à nos ECG ?
- PULSE-ECG/ECGInstruct (entraînement de PULSE, et repris par GEM et ECG-R1)
- PKUDigitalHealth/ECG-Protocol-Guided-Grounding-CoT (entraînement ECG-R1)
- LANSG/ECG-Grounding (entraînement GEM)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
PAT = re.compile(rb"(\d{5})_(?:hr|lr)\b")
def scan(path, chunk=1 << 26):
    ids, ctx, n_ptb = set(), [], 0
    tail = b""
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            buf = tail + b
            ids |= {int(m.group(1)) for m in PAT.finditer(buf)}
            n_ptb += len(re.findall(rb"ptb", buf, re.I))
            if len(ctx) < 5:
                for m in re.finditer(rb"ptb", buf, re.I):
                    ctx.append(buf[max(0, m.start() - 60):m.end() + 80].decode("utf8", "replace"));
                    if len(ctx) >= 5: break
            tail = buf[-200:]
    return ids, ctx, n_ptb
REPOS = ["PULSE-ECG/ECGInstruct", "PKUDigitalHealth/ECG-Protocol-Guided-Grounding-CoT", "LANSG/ECG-Grounding"]
fold = db.strat_fold
included = set(sel[sel.excluded == ""].ecg_id)
contam, ALL_IDS = [], {}
if MOCK:
    print("mode test : section F sautée")
else:
    api = HfApi()
    for repo in REPOS:
        try:
            files = [s for s in api.list_repo_tree(repo, repo_type="dataset", recursive=True)
                     if getattr(s, "size", None) and s.path.endswith((".json", ".jsonl")) and s.size < 4e9]
        except Exception as e:
            print(repo, ": liste impossible :", e); continue
        for s in files:
            try:
                p = hf_hub_download(repo, s.path, repo_type="dataset", cache_dir="/tmp/hf")
            except Exception as e:
                print(repo, s.path, ": téléchargement impossible :", e); continue
            ids, ctx, n_ptb = scan(p)
            ids = {i for i in ids if i in fold.index}
            f10 = {i for i in ids if fold[i] == 10}
            tag = (repo.split("/")[0] + "__" + s.path.replace("/", "__")).replace(".jsonl", "").replace(".json", "")
            json.dump(sorted(ids), open(f"{OUT}/ids__{tag}.json", "w"))
            ALL_IDS[tag] = ids
            contam.append(dict(repo=repo, file=s.path, size_MB=round(s.size / 1e6), ptb_mentions=n_ptb,
                               ptbxl_ids=len(ids), fold10_ids=len(f10), our_included_ids=len(ids & included),
                               folds=json.dumps(pd.Series([fold[i] for i in ids]).value_counts().sort_index().to_dict()) if ids else ""))
            print(contam[-1])
            if n_ptb and not ids:
                print("   'ptb' présent mais aucun identifiant reconnu ; exemples :", ctx[:3])
            os.remove(p)
    pd.DataFrame(contam).to_csv(f"{OUT}/contamination.csv", index=False)
    # union of PTB-XL records seen in training (instruction files, not benchmark/test files)
    train = set().union(*[v for k, v in ALL_IDS.items() if "train" in k.lower() or "ECGInstruct" in k or "Grounding_30k" in k])
    test = set().union(*[v for k, v in ALL_IDS.items() if "ptb-test" in k or "test-ptbxl" in k])
    json.dump(sorted(train), open(f"{OUT}/ptbxl_ids_in_training.json", "w"))
    print(f"\nPTB-XL vus à l'entraînement : {len(train)} | benchmark PTB-XL test : {len(test)} | "
          f"test ET entraînement : {len(train & test)} | nos ECG inclus hors entraînement : {len(included - train)}")
    print("fichier d'exclusion : calib/ptbxl_ids_in_training.json (à passer à run_ptbxl.py --exclude_ids)")

## Résumé à copier-coller à Claude

In [ ]:
print("=== A. quantiles (médiane [Q1-Q3]) ===")
for grp, d in real.groupby("group"):
    print(grp, "n=%d" % len(d), {c: f"{d[c].median():.3f} [{d[c].quantile(.25):.3f}-{d[c].quantile(.75):.3f}]" for c in cols})
print("\n=== B. éditions AF (médianes) ==="); print(g.round(3).to_string(index=False))
print("AF_SEVERITY proposé :", prop)
print("\n=== C. ROC ==="); print(roc_tab.round(3).to_string(index=False))
print("\n=== D. sélection fold 10 ==="); print(tab.to_string(index=False)); print(reasons.to_string())
print("\n=== E. 12SL ==="); print(agree.round(3).to_string(index=False) if agree is not None else "non disponible")
print("\n=== F. contamination ==="); print(pd.DataFrame(contam).to_string(index=False) if contam else "non disponible")
!cd /kaggle/working && zip -qr calib_resultats.zip calib
print("\nFichier : /kaggle/working/calib_resultats.zip (onglet Output)")